In [1]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))
import subprocess

import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper, PDFTableExtractor

helper = Helper()

In [ ]:
import requests
import pandas as pd

url = "https://api.bseindia.com/BseIndiaAPI/api/AnnSubCategoryGetData/w"

params = {
    "pageno": 1,
    "strCat": "Result",
    "strPrevDate": "20260401",
    "strScrip": "532174",
    "strSearch": "P",
    "strToDate": "20260521",
    "strType": "C",
    "subcategory": "-1"
}


headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.bseindia.com/",
    "Accept": "application/json"
}

bse_codes = [
    500112, 532215, 500247, 500180, 532174, 532134, 532483, 532461, 532477,
    532814, 539437, 532149, 540611, 532648, 500469, 532187, 532505, 532388,
    532525, 500116, 532885, 532210, 541153, 590003, 532209, 540065, 542904,
    532772, 543596, 532218, 542867, 544118, 543243, 532652, 544120, 533295,
    543942, 543386, 543279, 544020, 532180
]


alls = []
for code in bse_codes:

    params["strScrip"] = str(code)
    
    response = requests.get(url, params=params, headers=headers, verify=False)
    data = response.json()
    # print(response.status_code)

    # # Extract records (key usually 'Table')
    records = data.get("Table", [])

    # Convert to DataFrame
    df = pd.DataFrame(records)
    alls.append(df)

# Save as CSV
# df.to_csv("bse_data.csv", index=False)
final_df = pd.concat(alls, ignore_index=True)
final_df.to_csv("bse_data.csv", index=False)
print("Saved to bse_data.csv")

In [3]:
pdf_path = r"C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF.pdf"
pdf_path = r"TEXT_PDF.pdf"

In [ ]:
# ==============================
# 1. EXTRACT TEXT
# ==============================
helper = Helper()
text_data = helper.get_pdf_text(pdf_path)
print("TEXT SAMPLE:", text_data[0][:10])


In [ ]:
# ==============================
# 2. GET ALL BLOCKS + IMAGES
# ==============================
helper = Helper()
all_data = helper.get_all_pdf_data(pdf_path)
pprint.pprint(all_data[0])
# print(all_data[0])

In [ ]:
# ==============================
# 3. CLIPPED DATA (EDIT BBOX)
# ==============================
helper = Helper()
bboxes = [
    (0, 0, 300, 400),
    (100, 200, 400, 600)
]

try:
    clipped = helper.get_clipped_data(pdf_path, bboxes)
    print("CLIPPED SAMPLE:", clipped[0])
except Exception as e:
    print("Clipping skipped:", e)


In [ ]:
# ==============================
# 4. DRAW LINES + RECTS
# ==============================
lines = [
    ((50, 50), (300, 50)),
    ((100, 100), (400, 100))
]
rects = [(50, 50, 200, 300)]
pages = [1]

pdf_path = r"STR.pdf"
output_draw = pdf_path.replace(".pdf", "_drawn.pdf")
helper.draw_lines_on_pdf( pdf_path,lines,rects,pages,output_draw)



Modified PDF saved to: C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF_drawn.pdf


In [5]:
# ==============================
# 5. LINE BOUNDARIES
# ==============================
helper = Helper()
pdf_path = r"STR_bbox_mask.pdf"
output_path =helper.draw_boundaries_on_lines(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['STR_bbox_mask_line_hltd.pdf']>

In [5]:
# ==============================
# 6. BLOCK BOUNDARIES
# ==============================
helper = Helper()
output_path = helper.draw_boundaries_on_pdf(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['SAMPLE_block_highlighted.pdf']>

In [3]:
# ==============================
# 7. SPAN BOUNDARIES
# ==============================
helper = Helper()
pdf_path = r"KOT_bbox_mask.pdf"
output_path = helper.draw_span_boundaries(pdf_path)
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['KOT_bbox_mask_span_hltd.pdf']>

In [4]:
# ==============================
# 8. BBOX ONLY TEXT
# ==============================
helper = Helper()
pdf_path = r"STR.pdf"
output_path = helper.mask_outside_bboxes(pdf_path,[(4.1, 116.3, 268.64, 724.84)])
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['STR_bbox_mask.pdf']>

In [ ]:
# ==============================
# 9. SINGLE BBOX DRAW
# ==============================
bbox = (100, 100, 400, 400)
helper = Helper()
helper.draw_bboxes_on_pdf(pdf_path, bbox)



In [2]:
lines = [150.55802154541016, 160.51199951171876, 168.57695529335425, 176.9100082397461, 193.70262087308444, 219.67801666259766, 226.69800567626953, 235.1252767389471, 243.6998200850053, 262.1579895019531, 269.3879928588867, 278.0580012003581, 286.23799981011285, 295.12049674987793, 303.3060028076172, 311.7725469415838, 320.83799833409927, 330.197998046875, 337.5918438251202, 347.4779968261719, 354.23799981011285, 362.84999389648436, 371.8139953613281, 381.3179931640625, 389.5979919433594, 397.2060139973958, 405.6921117446002, 414.58800506591797, 422.84505866555605]
line_content = []
for line_y in lines:
    val = ((0,line_y),(800,line_y))
    line_content.append(val)

In [3]:
# lines = [
#     ((110, 0), (110, 812)),# Vertical line
#     ((0, 350), (812, 350)),
#     ((570, 0), (570, 812))
# ]
from pathlib import Path
from app.utils import Helper

lines  = line_content
pages = [2]
bboxes = []
sample_path = r"C:\Users\kaustubh.keny\Downloads\apollo_table 2.pdf"
file_name = Path(sample_path).name


Helper.draw_lines_on_pdf(sample_path, lines, bboxes, pages, file_name.replace(".pdf","_line.pdf"))

Modified PDF saved to: apollo_table 2_line.pdf


In [ ]:
import fitz
import pytesseract
from PIL import Image
import io, re

def get_proper_fund_names(path: str):
    title = {}
    pattern ="((?:LI?i?C|BSE|BANK|SMALL|HEALTH|MNEY|[aA]n\\s*open).*?(?:FUND|Path|ETF|FTF|EOF|FOF|PLAN|SAVER|tax saving scheme|small cap stocks)\\s*(?:FUND\\s*OF\\s*FUND)?)"
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            clip = fitz.Rect(300, 0, 595, 80)
            pix = page.get_pixmap(clip=clip, dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes()))
            text = pytesseract.image_to_string(img)
            cleaned = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            if matches := re.findall(pattern, cleaned, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _])
                print(f"{pgn}:matched {matches[0]}")
            # print(f"[OCR] Page {pgn}: {cleaned}")
            # if cleaned:
            #     title[pgn] = cleaned
    return title
path = r"C:\Users\kaustubh.keny\OneDrive - Cogencis Information Services Ltd\Documents\MUTUAL FUND FACTSHEET FY19-25\2021_changed\LIC Mutual Fund\25_31-Dec-21_FS.pdf"
title = get_proper_fund_names(path)

In [1]:
import re, fitz

def get_proper_fund_names(path: str, pattern:str,clip:tuple):
    title = {} 
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            text = " ".join(page.get_text("text", clip = clip).split("\n")) #clip = (0, 0, 210, 155)
            text = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            print(f"{pgn}:-{text}")
            if matches := re.findall(pattern, text, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _ ])
                print(pgn,matches[0])
    return title
path = r"94_30-Apr-26_IF.pdf"
pattern =  "FUND(.+?FUND)"
clip = (0, 0, 455, 70)
# pattern = "(MIRAE.*?)NSE\\s*[Ss]ymbol"
title = get_proper_fund_names(path,pattern,clip)

0:-
1:-
2:-
3:-FUND MANAGER DETAILS
4:-FUND MANAGER DETAILS
5:-Content Investment Report Equity Funds
6:-EQUITY FUND
7:-WHOLE LIFE MID CAP EQUITY FUND
8:-LARGE CAP EQUITY FUND
9:-FUTURE EQUITY PENSION FUND
10:-SELECT EQUITY FUND
11:-FUTURE SELECT EQUITY FUND
12:-TOP 0 FUND
13:-INFRASTRUCTURE FUND
14:-TOP 200 FUND
15:-SUPER SELECT EQUITY FUND
16:-SUPER SELECT EQUITY PENSION FUND
17:-INDIA CONSUMPTION FUND
18:-MULTI CAP FUND
19:-EMERGING OPPORTUNITIES FUND
20:-SUSTAINABLE EQUITY FUND
21:-SMALL CAP DISCOVERY FUND
22:-FLEXI GROWTH FUND
23:-RISING INDIA FUND
24:-MIDCAP MOMENTUM INDEX FUND
25:-FLEXI GROWTH FD II
26:-NIFTY ALPHA 0 INDEX FUND
27:-MULTICAP MOMENTUM QUALITY INDEX FUND
28:-ALPHA 0 INDEX PENSION FUND
29:-MULTICAP MOMENTUM QUALITY INDEX PENSION FUND
30:-TAX BONANZA CONSUMPTION FUND
31:-TAX BONANZA CONSUMPTION PENSION FUND
32:-TOP 200 ALPHA 30 INDEX FUND
33:-TOP 200 ALPHA 30 INDEX PENSION FUND
34:-INDEX FUND
35:-D RS INDEX FUND
36:-D RS INDEX PENSION FUND
37:-PPORTUNITIES FUND
38:-P

In [ ]:
def extract_clipped_data(input:str, pages:list, bboxes:list):
        
        document = fitz.open(input)
        final_list = []
    
        for pgn in pages:
            page = document[pgn]
            
            all_blocks = [] #store every data from bboxes
            
            for bbox in bboxes:
                blocks, seen_blocks = [], set()  #store unique blocks based on content and bbox
                
                page_blocks = page.get_text('dict', clip=bbox)['blocks']
                for block in page_blocks:
                    if block['type'] == 0 and 'lines' in block: #type 0 means text block
                        #hash_key
                        block_key = (tuple(block['bbox']), tuple(tuple(line['spans'][0]['text'] for line in block['lines'])))
                        if block_key not in seen_blocks:
                            seen_blocks.add(block_key)
                            blocks.append(block)

                sorted_blocks = sorted(blocks, key=lambda x: (x['bbox'][1], x['bbox'][0]))
                all_blocks.append(sorted_blocks)

            final_list.append({
                "pgn": pgn,
                "block": all_blocks #will be list[list,list,..]
            })

        document.close()
        return final_list
    
def extract_data_relative_line(path: str, line_x: float, side: str):
    doc = fitz.open(path)
    pages = doc.page_count

    final_list = []

    for pgn in range(pages):
        page = doc[pgn]

        blocks = page.get_text("dict")["blocks"]
        sorted_blocks = sorted(blocks, key=lambda x: (x["bbox"][1], x["bbox"][0]))
        extracted_blocks = []

        # Keep track of blocks to avoid duplicates
        added_blocks = set()

        for block in sorted_blocks:
            block_id = id(block)  # Unique identifier for the block

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    origin = span["origin"]
                    x0, _ = origin

                    # Check the side condition
                    if side == "left" and x0 < line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added
                    elif side == "right" and x0 > line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added

      
        final_list.append({
            "pgn": pgn,
            "blocks": extracted_blocks
        })

    doc.close()

    return final_list
  
def get_clipped_data(input:str, bboxes:list[set], *args):
    
        document = fitz.open(input)
        final_list = []
        if args:
            pages = list(args)
        else:
            pages = [i for i in document.page_count]
        
        for pgn in pages:
            page = document[pgn]

            blocks = []
            for bbox in bboxes:
                blocks.extend(page.get_text('dict', clip = bbox)['blocks']) #get all blocks
            
            filtered_blocks = [block for block in blocks if block['type']== 0 and 'lines' in block]
            # sorted_blocks = sorted(filtered_blocks, key= lambda x: (x['bbox'][1], x['bbox'][0]))
             # Extract text from sorted blocks
            extracted_text = []
            for block in filtered_blocks:
                block_text = []
                for line in block['lines']:
                    line_text = " ".join(span['text'] for span in line['spans'])
                    block_text.append(line_text)
                extracted_text.append("\n".join(block_text))
            
            final_list.append({
            "pgn": pgn,
            "block": filtered_blocks,
            "text": extracted_text
            })
            
            
        document.close()
        return final_list
    
def extract_clipped_text_all_pages(pdf_path, clip_coords):
    results = {}
    doc = fitz.open(pdf_path)
    clip_rect = fitz.Rect(*clip_coords)
    try:
        for page_number, page in enumerate(doc):
            text = page.get_text("text", clip=clip_rect).strip()
            results[page_number] = text
    finally:
        doc.close()
    return results

In [6]:
import pandas as pd
import os

from app.parse_table import TableParser


parser = TableParser()

path = r"C:\Users\kaustubh.keny\Downloads\disclosure\Canara Robeco Mutual Fund"

all_data = []

for file in os.listdir(path):
    if not file.endswith(".pdf"):
        continue
    
    pdf_path = os.path.join(path,file)
    
    print(file)
    df = parser.extract_tables_from_pdf(path=pdf_path,pages=None)
    # print(df.shape)
    
    old_cols = list(df.columns)
    
    df["filename"] = file
    
    new_cols = ["filename"] + old_cols
    df = df.reindex(columns = new_cols)
    all_data.append(df)
    
d1 = pd.concat(all_data, axis=0, ignore_index=True)
d1.to_excel("CAN_FINAL.xlsx")

Disclosure-of-Commission 2012-13.pdf
Disclosure-of-Commission 2013-14.pdf
Disclosure-of-Commission 2014-15.pdf
Disclosure-of-Commission 2015-16_.pdf
Disclosure-of-Commission 2016-17.pdf
Disclosure-of-Commission 2017-18.pdf
Disclosure-of-Commission 2018-19.pdf
Disclosure-of-Commission 2019-20.pdf
Disclosure-of-Commission 2020-21.pdf
Disclosure-of-Commission 2021-22.pdf
Disclosure-of-Commission 2022-23.pdf
Disclosure-of-Commission 2023-24.pdf
Disclosure-of-Commission 2024-25.pdf


0